# Autoship Nudge Promo Incentive — Data Analysis (Demand-Event Definition)

This notebook builds up, CTE by CTE, the warehouse query used to establish the Autoship Adoption Rate baseline in `power_analysis_aar_v1.ipynb`. Each step below adds one CTE and re-runs, so the final steps reproduce the exact queries used for sizing.

**Population:** Manual clients (i.e., not already enrolled in Autoship) who completed First Fix checkout with a Buy 1+ keep rate — the eligible population for the Autoship Nudge Promo Incentive Test, per the experiment's PRD.

**Source tables:**
- `curated.merch_sales_and_feedback`, an item-level Fix/direct-buy fact table, used to identify each client's First Fix, its keep rate, and whether it was fulfilled manually or under Autoship.
- `curated.client_pulse_journal`, a daily client-state journal, used to read the Autoship adoption signal itself: `last_autoship_demand_ts`, the timestamp of a client's most recent Autoship demand event as of that journal row.

In [1]:
import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

## Part A — Identifying each client's First Fix

`curated.merch_sales_and_feedback` is at the item grain (one row per item shipped), so a client's First Fix (`fix_number = 1`) spans multiple rows. This part collapses those rows to one row per client's first-Fix shipment, carrying forward the shipment's `autoship_or_manual` value and the count of items kept.

### A1 — raw item-level rows, `fix_number = 1`

In [2]:
query("""--sql
SELECT client_id, shipment_id, item_id, checkout_date, autoship_or_manual, sold_paid_fix_flag, business_line
FROM curated.merch_sales_and_feedback
WHERE fix_number = 1
  AND created_date >= DATE '2026-06-01'
ORDER BY client_id, shipment_id
LIMIT 8
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,shipment_id,item_id,checkout_date,autoship_or_manual,sold_paid_fix_flag,business_line
0,3008420,134765685,388424823,2026-07-05,autoship,0,Womens
1,3008420,134765685,387578192,2026-07-05,autoship,0,Womens
2,3008420,134765685,372520741,2026-07-05,autoship,0,Womens
3,3008420,134765685,388218846,2026-07-05,autoship,0,Womens
4,3008420,134765685,379180269,2026-07-05,autoship,0,Womens
5,3010612,134862228,387289333,2026-07-11,autoship,0,Womens
6,3010612,134862228,388574655,2026-07-11,autoship,0,Womens
7,3010612,134862228,385837912,2026-07-11,autoship,0,Womens


Each client's first-Fix shipment shows up as several rows (one per item). `autoship_or_manual` is constant within a shipment, so it collapses cleanly with `ARBITRARY()`; `sold_paid_fix_flag` is summed per shipment to get the number of items kept.

### A2 — collapse to one row per (client, shipment), and check cardinality

In [3]:
query("""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-01-01'
    GROUP BY client_id, shipment_id
)
SELECT n_shipments, COUNT(*) AS n_clients
FROM (
    SELECT client_id, COUNT(DISTINCT shipment_id) AS n_shipments
    FROM first_fix
    GROUP BY client_id
)
GROUP BY n_shipments
ORDER BY n_shipments
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_shipments,n_clients
0,1,389721
1,2,538
2,3,17
3,4,7
4,6,1


The overwhelming majority of clients have exactly one shipment tagged `fix_number = 1`. The final query below keeps only the chronologically earliest such shipment per client via `ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date)`, so every eligible client contributes exactly one row.

## Part B — Manual vs. Autoship split, and why a recent cohort is used

`autoship_or_manual` marks whether a Fix was fulfilled under an active Autoship subscription at the time. Historically, clients could enroll in Autoship at signup, before ever receiving a Fix — so a meaningful share of *First* Fixes were historically already `'autoship'`. A more recent rollout moved Autoship enrollment to **after** First Fix checkout, once keep rate is known, via a post-checkout nudge. That shifts the manual share of First Fixes upward over time, which is why the baseline later in this notebook uses a recent reference month rather than a long historical average.

In [4]:
query("""--sql
SELECT
    DATE_TRUNC('month', created_date) AS month,
    autoship_or_manual,
    COUNT(DISTINCT client_id) AS n_clients
FROM curated.merch_sales_and_feedback
WHERE fix_number = 1
  AND created_date >= DATE '2025-08-01'
GROUP BY 1, 2
ORDER BY 1 DESC, 2
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,autoship_or_manual,n_clients
0,2026-08-01,autoship,4729
1,2026-08-01,manual,5415
2,2026-07-01,autoship,24586
3,2026-07-01,manual,28476
4,2026-06-01,autoship,20647
5,2026-06-01,manual,19868
6,2026-05-01,autoship,36903
7,2026-05-01,manual,12892
8,2026-04-01,autoship,42355
9,2026-04-01,manual,14193


## Part C — Keep rate: Buy 0 vs. Buy 1+

The PRD scopes this test to clients with a **Buy 1+** keep rate on their First Fix (kept at least one item) — Buy 0 clients always get the BAU Quick Fix experience with no Autoship nudge at all, and are out of scope for this promo/non-promo comparison. `sold_paid_fix_flag`, summed per shipment, gives the number of items kept.

In [5]:
query("""--sql
WITH first_fix AS (
    SELECT client_id, shipment_id, SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-06-01'
    GROUP BY client_id, shipment_id
)
SELECT n_items_kept, COUNT(*) AS n_shipments
FROM first_fix
GROUP BY n_items_kept
ORDER BY n_items_kept
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_items_kept,n_shipments
0,0,42573
1,1,12026
2,2,11896
3,3,9547
4,4,4744
5,5,14172
6,6,1563
7,7,980
8,8,1955
9,9,236


Roughly half to three-fifths of recent First Fixes keep at least one item — the Buy 1+ gate this test's population applies.

## Part D — Introducing `client_pulse_journal`: a daily client-state journal

`curated.client_pulse_journal` is a slowly-changing-dimension journal: one row per day any tracked client attribute changes, bounded by `start_date`/`end_date`, ordered by `sequence`, with `is_current` marking the row that's still open. Two of its columns carry an Autoship signal directly: `last_autoship_demand_ts` (the timestamp of the client's most recent Autoship *demand* event — i.e., when an Autoship subscription was created or renewed) and `last_autoship_checkout_ts` (the timestamp of the most recent Fix *checked out* under that subscription).

The gap between these two matters: demand happens when a client enrolls, checkout happens later when that enrollment's next Fix actually ships. The example below is a real client's full journal history, picked because their First Fix checkout (per Part A/C's population logic) falls on **2026-02-11**.

In [6]:
query("""--sql
SELECT client_id, sequence, start_date, end_date, is_current, last_autoship_demand_ts, last_autoship_checkout_ts
FROM curated.client_pulse_journal
WHERE client_id = 5091131
ORDER BY sequence
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sequence,start_date,end_date,is_current,last_autoship_demand_ts,last_autoship_checkout_ts
0,5091131,1,2020-04-01 00:00:00.000,2024-09-13 00:00:00.000,0,None,None
1,5091131,2,2024-09-13 00:00:00.000,2025-05-08 00:00:00.000,0,None,None
2,5091131,3,2025-05-08 00:00:00.000,2025-12-17 00:00:00.000,0,None,None
3,5091131,4,2025-12-17 00:00:00.000,2026-02-05 00:00:00.000,0,None,None
4,5091131,5,2026-02-05 00:00:00.000,2026-02-08 00:00:00.000,0,None,None
5,5091131,6,2026-02-08 00:00:00.000,2026-02-09 00:00:00.000,0,None,None
6,5091131,7,2026-02-09 00:00:00.000,2026-02-11 00:00:00.000,0,None,None
7,5091131,8,2026-02-11 00:00:00.000,2026-02-12 00:00:00.000,0,None,None
8,5091131,9,2026-02-12 00:00:00.000,2026-02-13 00:00:00.000,0,None,None
9,5091131,10,2026-02-13 00:00:00.000,2026-02-14 00:00:00.000,0,None,None


**Reading this:** `last_autoship_demand_ts` is `None` through the row covering this client's First Fix checkout (2026-02-11), then appears for the first time on **2026-02-14** — 3 days after checkout, consistent with a nudge-driven opt-in. It then stays frozen at that same February value for the next several months while the subscription's future Fixes queue up, and the corresponding `last_autoship_checkout_ts` doesn't appear until **2026-06-02** — almost 4 months after the demand event, when that first Autoship-driven Fix finally ships. A definition that waited for `last_autoship_checkout_ts` (i.e., for the next Fix to actually resolve) would report this client's adoption 4 months later than it actually happened, and would report nothing at all for a client who adopted but cancelled before ever reaching checkout.

## Part E — Why the demand timestamp must be compared against checkout, not just "populated"

Not every Autoship demand timestamp reflects this test's nudge. Some clients carry a demand event from **before** their First Fix entirely — a historical or otherwise-unrelated Autoship enrollment. The example below is another real client, whose First Fix checkout falls on **2026-02-14**.

In [7]:
query("""--sql
SELECT client_id, sequence, start_date, end_date, is_current, last_autoship_demand_ts, last_autoship_checkout_ts
FROM curated.client_pulse_journal
WHERE client_id = 11189128
ORDER BY sequence
LIMIT 20
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,sequence,start_date,end_date,is_current,last_autoship_demand_ts,last_autoship_checkout_ts
0,11189128,1,2020-04-01 00:00:00.000,2025-05-22 00:00:00.000,0,None,None
1,11189128,2,2025-05-22 00:00:00.000,2025-05-24 00:00:00.000,0,2025-05-22 12:27:33.000,None
2,11189128,3,2025-05-24 00:00:00.000,2025-07-15 00:00:00.000,0,2025-05-22 12:27:33.000,None
3,11189128,4,2025-07-15 00:00:00.000,2025-07-17 00:00:00.000,0,2025-05-22 12:27:33.000,None
4,11189128,5,2025-07-17 00:00:00.000,2025-09-08 00:00:00.000,0,2025-05-22 12:27:33.000,None
5,11189128,6,2025-09-08 00:00:00.000,2025-09-10 00:00:00.000,0,2025-05-22 12:27:33.000,None
6,11189128,7,2025-09-10 00:00:00.000,2025-11-03 00:00:00.000,0,2025-05-22 12:27:33.000,None
7,11189128,8,2025-11-03 00:00:00.000,2025-11-05 00:00:00.000,0,2025-05-22 12:27:33.000,None
8,11189128,9,2025-11-05 00:00:00.000,2026-02-08 00:00:00.000,0,None,None
9,11189128,10,2026-02-08 00:00:00.000,2026-02-13 00:00:00.000,0,2026-02-08 23:44:34.000,None


**Reading this, two things stand out.** First, this client already has a demand timestamp from **2026-02-08** — 6 days *before* their First Fix checkout on 2026-02-14 — which later converts into a checkout on 2026-03-12. Counting that as a fresh, nudge-driven adoption would be wrong: the demand predates the very Fix this test's nudge is shown after. Requiring the demand timestamp to be strictly *after* First Fix checkout excludes this case correctly.

Second, look at rows 2-8 versus row 9: an *even earlier* demand timestamp from 2025-05-22 is present, then **disappears back to `None`** at the row starting 2025-11-05, before the 2026-02-08 timestamp appears. `last_autoship_demand_ts` gets reset when the underlying subscription is fully cancelled — which is exactly why this notebook's query scans a client's **entire** journal history for the earliest qualifying value, rather than reading only today's (`is_current = '1'`) row. A snapshot-only read would already have missed the 2025-05-22 episode by the time this table is queried; scanning full history doesn't have that blind spot.

## Part F — Quantifying the difference: full history vs. today's snapshot

To confirm this isn't just a two-client curiosity, this step compares, across a real cohort, how many clients show a fresh post-First-Fix demand signal when reading only today's row versus scanning full history.

In [8]:
query("""--sql
WITH first_fix AS (
    SELECT client_id, shipment_id, MIN(checkout_date) AS checkout_date_1,
           ARBITRARY(autoship_or_manual) AS autoship_or_manual, SUM(sold_paid_fix_flag) AS n_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1 AND created_date >= DATE '2026-01-01' AND created_date < DATE '2026-03-01'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date_1
    FROM first_fix
    WHERE autoship_or_manual = 'manual' AND n_kept >= 1
),
pulse_current AS (
    SELECT client_id, last_autoship_demand_ts
    FROM curated.client_pulse_journal
    WHERE is_current = '1'
),
pulse_full_history AS (
    SELECT e.client_id, MIN(p.last_autoship_demand_ts) AS first_fresh_demand_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id AND p.last_autoship_demand_ts > e.checkout_date_1
    GROUP BY e.client_id
)
SELECT
    COUNT(*) AS n_eligible,
    SUM(CASE WHEN c.last_autoship_demand_ts IS NOT NULL AND c.last_autoship_demand_ts > e.checkout_date_1 THEN 1 ELSE 0 END) AS n_adopted_snapshot_only,
    (SELECT COUNT(*) FROM pulse_full_history) AS n_adopted_full_history
FROM eligible e
LEFT JOIN pulse_current c ON c.client_id = e.client_id
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_eligible,n_adopted_snapshot_only,n_adopted_full_history
0,18753,4091,4566


**Reading this:** the full-history read finds more adopters than the snapshot-only read — consistent with Part E's cancellation-reset case being a real, non-trivial pattern in this population, not an isolated example. This notebook's baseline query (Part G) uses the full-history version throughout.

## Part G — The full baseline query

Putting it together: `eligible` applies the Manual + Buy 1+ gate (deduped to each client's earliest `fix_number = 1` shipment) and a 90-day maturation cutoff so the adoption read has had time to resolve; `fresh_demand` finds each eligible client's earliest post-First-Fix Autoship demand timestamp, scanning full journal history per Parts D-F; the final `SELECT` flags `adopted_autoship` and aggregates to a monthly Autoship Adoption Rate and daily eligible volume. This is the exact query used for this experiment's Autoship Adoption Rate sizing.

In [9]:
MATURATION_DAYS = 90
COHORT_START = '2025-08-01'

baseline_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
      AND checkout_date <= CURRENT_DATE - INTERVAL '{MATURATION_DAYS}' DAY
),
fresh_demand AS (
    SELECT e.client_id, MIN(p.last_autoship_demand_ts) AS first_fresh_demand_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id
     AND p.last_autoship_demand_ts > e.checkout_date
    GROUP BY e.client_id
),
joined AS (
    SELECT
        e.client_id,
        DATE_TRUNC('month', e.checkout_date) AS month,
        e.checkout_date,
        CASE WHEN f.first_fresh_demand_ts IS NOT NULL
              AND f.first_fresh_demand_ts <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
             THEN 1 ELSE 0 END AS adopted_autoship
    FROM eligible e
    LEFT JOIN fresh_demand f ON f.client_id = e.client_id
),
month_days AS (
    SELECT month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM joined
    GROUP BY month
)
SELECT
    j.month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    SUM(adopted_autoship) AS n_adopted,
    CAST(SUM(adopted_autoship) AS DOUBLE) / COUNT(*) AS autoship_adoption_rate,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM joined j
JOIN month_days md ON j.month = md.month
GROUP BY j.month, md.days_observed
ORDER BY j.month DESC
"""

query(baseline_query)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,n_adopted,autoship_adoption_rate,eligible_per_day
0,2026-05-01,8,2679,701,0.261665,334.9
1,2026-04-01,30,10260,2427,0.236550,342.0
2,2026-03-01,31,10857,2350,0.216450,350.2
3,2026-02-01,28,8800,2105,0.239205,314.3
4,2026-01-01,31,10365,2699,0.260396,334.4
5,2025-12-01,31,8915,2983,0.334605,287.6
6,2025-11-01,30,7250,1462,0.201655,241.7
7,2025-10-01,31,9201,1581,0.171829,296.8
8,2025-09-01,30,9272,1471,0.158650,309.1
9,2025-08-01,31,7167,1031,0.143854,231.2


**Reference month:** the most recent fully-mature calendar month (all its days past the 90-day maturation cutoff) is the reference month used for the adoption-rate baseline.

## Part H — A fresher, decoupled daily-volume read

Daily eligible volume doesn't need the 90-day maturation wait the adoption-rate read needs — whether a client is Manual + Buy 1+ is known immediately at First Fix checkout. That means volume can be measured off the **most recent complete month**, even though that same month is too recent to supply a matured adoption-rate read.

In [10]:
query("""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '2026-05-01'
    GROUP BY client_id, shipment_id
),
eligible AS (
    SELECT client_id, checkout_date
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
      AND autoship_or_manual = 'manual'
      AND n_items_kept >= 1
),
month_days AS (
    SELECT DATE_TRUNC('month', checkout_date) AS month, COUNT(DISTINCT checkout_date) AS days_observed
    FROM eligible
    GROUP BY 1
)
SELECT
    DATE_TRUNC('month', e.checkout_date) AS month,
    md.days_observed,
    COUNT(*) AS n_eligible,
    ROUND(COUNT(*) / CAST(md.days_observed AS DOUBLE), 1) AS eligible_per_day
FROM eligible e
JOIN month_days md ON DATE_TRUNC('month', e.checkout_date) = md.month
GROUP BY DATE_TRUNC('month', e.checkout_date), md.days_observed
ORDER BY 1 DESC
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,n_eligible,eligible_per_day
0,2026-08-01,5,2661,532.2
1,2026-07-01,31,17749,572.5
2,2026-06-01,30,12149,405.0
3,2026-05-01,31,6821,220.0
4,2026-04-01,1,10,10.0


**Reading this:** the most recent complete month runs meaningfully higher than the matured reference month used for the rate — consistent with Part B's observation that the eligible population is still growing as a share of all First Fixes. Using this more recent month's volume for duration calculations, decoupled from the rate, avoids understating the run rate with a stale volume figure.